# Sovereign-Mahjong V6 — Kaggle End-to-End Pipeline (Convert + Phase 0 + Phase 1/2 RL + Phase 3 SD-TAP + Strict Eval)

本 Notebook 专为 **Kaggle GPU（T4 / P100 / L4 / H100）** 优化，完整覆盖 **Sovereign-Mahjong V6** 从 2026 天凤凤桌牌谱下载、GPU 无损批转换，到 Phase 0-A/B/C、Phase 1/2 PureJaxRL、Phase 3 SD-TAP 行为蒸馏与严格 Duplicate 1v3 评测的全流程。

### 核心数学硬门控（全流程强制校验）
1. **数据零损失**：CPU 精确 136 牌山重建 + GPU `vmap(lax.scan)` 批重放 `mahjax` 规则引擎（自动补全隐式 `PASS` 与 6 类流局，`float16` 无损存储 `{0, 1, k/4, k/8}` 空间特征，约 `235 B/步`）。
2. **Phase 0-B (`GRP_base`)**：10 步 `log_sinkhorn_4x4` Birkhoff 误差 `< 1e-4`、凤桌 7d 精确零和误差 `< 1e-4`、留出集 `ECE < 0.04`；训后严格冻结作为 DAG 防火墙。
3. **Phase 0-A (`BeliefNet` + `T_cal`)**：36× 等变（$S_3^{	ext{suit}} 	imes S_3^{	ext{sangen}}$）+ 严格**无加权**先验偏置 BCE（`-4.184`）+ 独立留出集温度校准 `T_cal`；硬门控：`ROC-AUC >= 0.85`、`mean(p_cal)/mean(y_ron) ≈ 1.00`。
4. **Phase 0-C（严格同构 `SovereignPolicyNet` BC）**：冻结 `B_EMA(T_cal) + GRP_base` 注入解析物理三联体，`beta_risk / beta_offense` 豁免权重衰减端到端学习。
5. **Phase 1/2 (`train_rl_v6`) & Phase 3 (`distill_sdtap`)**：冻策略预热 + Per-Rollout 单快照零重编译 PFSP + 双速率 Simplex TD($\lambda$) + 无偏 Running-RMS（不减均值）+ 外循环重置仅清零 AdamW `m_t` 完整保留 `v_t` + 三重护甲安全盾 + 自愈熔断 + 行为空间自蒸馏。
6. **严格评测标准**：固定牌山全半庄 Duplicate 1v3（`round_mode="half"`, `round_end="confirm"`，4 席轮换配对 95% 置信区间，天凤凤桌 7d `[+90, +45, 0, -135]`）+ 零样本多规则切换（`7d / 10d / M-League`）。

> **Kaggle 磁盘提示**：原始 451 MB Parquet 放在 `/kaggle/tmp/`（不占 20 GB 持久化输出配额）；转换后的分片与所有模型权重/报表持久化保存在 `/kaggle/working/`。

## 1. 环境检查、拉取仓库与依赖安装

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"

# 1. Inspect GPU
!nvidia-smi

# 2. Clone repository (or reuse existing /kaggle/working/mahjax)
REPO_URL = "https://github.com/myouo/mahjax.git"
REPO_BRANCH = "main"
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
TMP_ROOT = Path("/kaggle/tmp") if Path("/kaggle/tmp").exists() else (WORK_ROOT / "tmp")
TMP_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = WORK_ROOT / "mahjax"
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

# 3. Install required packages (keep Kaggle's preinstalled CUDA JAX if already working)
def _ensure_pkg(import_name: str, pip_spec: str) -> None:
    try:
        __import__(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_spec], check=True)

_ensure_pkg("jax", "jax[cuda12]")
_ensure_pkg("flax", "flax>=0.10.0")
_ensure_pkg("optax", "optax>=0.2.4")
_ensure_pkg("pyarrow", "pyarrow")
_ensure_pkg("huggingface_hub", "huggingface_hub")

# Editable install of mahjax + mahjong_rl
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
# Evict any stale mahjong_rl modules from Jupyter sys.modules after git pull
for _mod in [m for m in list(sys.modules) if m == "mahjong_rl" or m.startswith("mahjong_rl.")]:
    del sys.modules[_mod]

import jax
print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())
assert any(d.platform == "gpu" or d.platform == "tpu" for d in jax.devices()), (
    "WARNING: No GPU/TPU accelerator detected! Enable GPU (T4 / P100) in Kaggle Notebook Settings."
)

## 2. 全局超参、路径与 Kaggle 预算配置

In [ ]:
import subprocess
from pathlib import Path
import jax

# ==============================================================================
# Auto-Detect Multi-GPU Count & Per-GPU VRAM (saturates Kaggle T4x2 / P100 / L4 / H100!)
# ==============================================================================
N_DEVICES = jax.local_device_count()
try:
    vram_lines = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
        text=True,
    ).strip().splitlines()
    VRAM_MB_PER_GPU = int(min(int(x.strip()) for x in vram_lines if x.strip()))
except Exception:
    VRAM_MB_PER_GPU = 15000

VRAM_GB = VRAM_MB_PER_GPU / 1024.0

if VRAM_GB >= 22.0:
    _PER_DEV_CONV_B, _PER_DEV_BEL_B, _PER_DEV_BC_B, _PER_DEV_RL_ENVS, _PER_DEV_RL_MB, _EVAL_CHUNK = 384, 1536, 768, 96, 1536, 16
elif VRAM_GB >= 14.0:
    # Kaggle T4 (15 GB x 2) / P100 (16 GB) saturated preset (~10-13 GB active VRAM per GPU!)
    _PER_DEV_CONV_B, _PER_DEV_BEL_B, _PER_DEV_BC_B, _PER_DEV_RL_ENVS, _PER_DEV_RL_MB, _EVAL_CHUNK = 256, 1024, 512, 64, 1024, 16
else:
    # 6-12 GB single GPU fallback
    _PER_DEV_CONV_B, _PER_DEV_BEL_B, _PER_DEV_BC_B, _PER_DEV_RL_ENVS, _PER_DEV_RL_MB, _EVAL_CHUNK = 96, 512, 512, 32, 512, 8

CFG = {
    # Stage switches (set False to reuse existing artifacts from /kaggle/input or /kaggle/working)
    "RUN_DOWNLOAD": True,
    "RUN_CONVERT": True,
    "RUN_PHASE0": True,
    "RUN_PHASE12_RL": True,
    "RUN_PHASE3_SDTAP": True,
    "RUN_DUPLICATE_EVAL": True,

    # Multi-GPU Pipelined Conversion (pmap across ALL GPUs + GPU float16 + zlib L3 + CPU prefetch!)
    # On Kaggle T4x2: 512 kyokus in parallel -> ~4-6x faster than single-GPU default!
    "START_GAME": 0,
    "MAX_GAMES": 10000,
    "FORCE_RECONVERT": False,                   # True -> delete existing shards and reconvert from scratch
    "ALLOW_PARTIAL_SHARDS": False,              # False -> auto-RESUME incomplete shards until MAX_GAMES is reached
    "ROUND_BATCH_SIZE": _PER_DEV_CONV_B * N_DEVICES,
    "MAX_STEPS_PER_ROUND": 144,                 # Covers >99.98% of Phoenix kyokus (25% less padding!)
    "SHARD_SIZE": 50000,
    "RON_K_MAX": _PER_DEV_CONV_B * 80,          # Per-GPU static compaction budget (0 overflows)
    "COMPRESS_LEVEL": 3,                        # Fast compact zlib level 3 (~3x faster, ~14MB/shard)

    # Optional: reuse pre-converted shards/checkpoints from a mounted Kaggle Dataset
    "PREEXISTING_SHARD_DIR": "",
    "PREEXISTING_CKPT_DIR": "",

    # Phase 0 offline supervised hyperparameters (auto-scaled across ALL GPUs via pmap!)
    "VAL_SHARDS": 4,
    "GRP_EPOCHS": 10,
    "GRP_BATCH_SIZE": 2048,
    "BELIEF_EPOCHS": 5,
    "BELIEF_BATCH_SIZE": _PER_DEV_BEL_B * N_DEVICES,
    "BC_EPOCHS": 5,
    "BC_BATCH_SIZE": _PER_DEV_BC_B * N_DEVICES,

    # Phase 1/2 Multi-GPU PureJaxRL (pmap rollouts + synchronized pmean grads across ALL GPUs!)
    "RL_NUM_ENVS": _PER_DEV_RL_ENVS * N_DEVICES,
    "RL_NUM_STEPS": 128,
    "RL_WARMUP_UPDATES": 20,
    "RL_NUM_UPDATES": 200,
    "RL_OUTER_RESET_STEPS": 100,
    "RL_MINIBATCH_SIZE": _PER_DEV_RL_MB * N_DEVICES,
    "RL_UPDATE_EPOCHS": 4,

    # Phase 3 SD-TAP behavioral self-distillation
    "SDTAP_EPOCHS": 2,
    "SDTAP_BATCH_SIZE": 256,

    # Strict Duplicate 1v3 evaluation (pmap across ALL GPUs: EVAL_CHUNK_WALLS * N_DEVICES walls/chunk!)
    "EVAL_WALLS": 32,
    "EVAL_CHUNK_WALLS": _EVAL_CHUNK,

    # Optional: auto-push checkpoints & reports to Hugging Face Hub (Kaggle secret HF_TOKEN)
    "HF_UPLOAD_REPO": "",
    "SEED": 0,
}

PARQUET_DIR = TMP_ROOT / "tenhou_2026_mjai"
BASELINE_DIR = TMP_ROOT / "baselines" / "tanyaodojo"
SHARD_DIR = Path(CFG["PREEXISTING_SHARD_DIR"]) if CFG["PREEXISTING_SHARD_DIR"] else (WORK_ROOT / "v6_shards_2026")
CKPT_DIR = WORK_ROOT / "checkpoints" / "sovereign_v6"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Detected Hardware: {N_DEVICES}x GPU ({VRAM_GB:.1f} GB/GPU = {N_DEVICES*VRAM_GB:.1f} GB total VRAM)")
print(f"Auto-Tuned Plan:   conv_batch={CFG['ROUND_BATCH_SIZE']} | belief_bs={CFG['BELIEF_BATCH_SIZE']} | bc_bs={CFG['BC_BATCH_SIZE']} | rl_envs={CFG['RL_NUM_ENVS']} | rl_mb={CFG['RL_MINIBATCH_SIZE']}")
print("SHARD_DIR:", SHARD_DIR)
print("CKPT_DIR: ", CKPT_DIR)

## 3. 下载 2026 天凤凤桌 MJAI Parquet 数据集与 TanyaoDojo 参考基线

In [ ]:
import subprocess
import pyarrow.parquet as pq

def _download(url: str, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 1024:
        print(f"  [skip] {dst.name} already exists ({dst.stat().st_size / 1e6:.1f} MB)")
        return
    print(f"  [fetch] {dst.name} ...")
    subprocess.run(["curl", "-fL", "--retry", "3", "-C", "-", url, "-o", str(dst)], check=True)

if CFG["RUN_DOWNLOAD"]:
    print("=== Downloading 2026 Tenhou Phoenix MJAI Parquet (hhim8826/tenhou-houou-mjai) ===")
    for fname in ("tenhou-new-00000.parquet", "tenhou-new-00001.parquet"):
        _download(
            f"https://huggingface.co/datasets/hhim8826/tenhou-houou-mjai/resolve/main/data/{fname}",
            PARQUET_DIR / fname,
        )

    print("\n=== Downloading Twu31/TanyaoDojo Reference Baseline (rl_oracle_800m.pkl) ===")
    for fname in ("README.md", "rl_oracle_800m.pkl"):
        _download(
            f"https://huggingface.co/Twu31/TanyaoDojo/resolve/main/{fname}",
            BASELINE_DIR / fname,
        )

    total_rows = 0
    for p in sorted(PARQUET_DIR.glob("*.parquet")):
        n = pq.ParquetFile(p).metadata.num_rows
        total_rows += n
        print(f"  Verified {p.name}: {n:,} games ({p.stat().st_size / 1e6:.1f} MB)")
    print(f"Total available 4p Phoenix hanchan games: {total_rows:,}")

## 4. GPU 批量无损转换：MJAI Parquet → Sovereign-Mahjong V6 `.npz` 分片

In [ ]:
import json
from mahjong_rl.data.mjai_parquet_converter import convert_mjai_parquets_to_v6_shards

existing_shards = sorted(SHARD_DIR.glob("shard_*.npz"))
manifest_path = SHARD_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else None

target_games = CFG["MAX_GAMES"]
manifest_complete = (
    manifest is not None
    and bool(manifest.get("completed", True))
    and int(manifest.get("start_game", 0)) == int(CFG["START_GAME"])
    and (target_games is None or int(manifest.get("games_processed", 0)) >= int(target_games))
    and len(existing_shards) >= max(len(manifest.get("shards_written", [])), 1)
)

need_convert = (
    CFG["RUN_CONVERT"]
    and (CFG.get("FORCE_RECONVERT", False) or not manifest_complete)
    and not (CFG.get("ALLOW_PARTIAL_SHARDS", False) and bool(existing_shards))
)

if need_convert:
    if CFG.get("FORCE_RECONVERT", False) and (existing_shards or manifest_path.exists()):
        print(f"[Convert] FORCE_RECONVERT=True — removing {len(existing_shards)} existing shard(s) ...")
        for _p in list(SHARD_DIR.glob("shard_*.npz")) + list(SHARD_DIR.glob(".shard_*.tmp.npz")):
            _p.unlink(missing_ok=True)
        manifest_path.unlink(missing_ok=True)
        (SHARD_DIR / "progress.json").unlink(missing_ok=True)
    elif existing_shards:
        print(
            f"[Convert] Keeping {len(existing_shards)} existing shard(s) and RESUMING automatically "
            f"from where conversion stopped until MAX_GAMES={target_games} ..."
        )
    parquets = sorted(PARQUET_DIR.glob("*.parquet"))
    assert parquets, f"No parquet files found in {PARQUET_DIR}"
    conv_res = convert_mjai_parquets_to_v6_shards(
        parquets,
        out_dir=SHARD_DIR,
        max_games=CFG["MAX_GAMES"],
        start_game=CFG["START_GAME"],
        hanchan_only=True,
        round_batch_size=CFG["ROUND_BATCH_SIZE"],
        max_steps_per_round=CFG["MAX_STEPS_PER_ROUND"],
        ron_k_max=CFG["RON_K_MAX"],
        shard_size=CFG["SHARD_SIZE"],
        storage_dtype="float16",
        include_oracle=True,
        compress=True,
        compresslevel=CFG["COMPRESS_LEVEL"],
        multi_gpu=True,
        resume=not CFG.get("FORCE_RECONVERT", False),
        verbose=True,
    )
    print("\nConversion Summary:")
    print(json.dumps({k: v for k, v in conv_res.items() if k != "shards_written"}, indent=2))
    assert conv_res["clean_round_rate"] >= 0.995, f"Unexpected desync rate: {conv_res['clean_round_rate']}"
    assert conv_res["ron_overflow_batches"] == 0, (
        f"ron_k_max overflowed {conv_res['ron_overflow_batches']} times; increase CFG['RON_K_MAX']!"
    )
elif manifest_complete:
    print(
        f"Using {len(existing_shards)} verified V6 shards from {SHARD_DIR} "
        f"(games={manifest.get('games_processed')}, steps={manifest.get('steps_written'):,}, "
        f"clean={manifest.get('clean_round_rate', 0.0)*100:.2f}%)"
    )
else:
    print(
        f"WARNING: Using {len(existing_shards)} partial/unverified shards from {SHARD_DIR} "
        f"(RUN_CONVERT={CFG['RUN_CONVERT']}, ALLOW_PARTIAL_SHARDS={CFG.get('ALLOW_PARTIAL_SHARDS', False)})"
    )

all_shards = sorted(SHARD_DIR.glob("shard_*.npz"))
assert len(all_shards) > CFG["VAL_SHARDS"], (
    f"Found only {len(all_shards)} shards; reduce VAL_SHARDS or increase MAX_GAMES."
)
train_shards = all_shards[: -CFG["VAL_SHARDS"]]
val_shards = all_shards[-CFG["VAL_SHARDS"] :]
print(f"Strict split: {len(train_shards)} train shards | {len(val_shards)} held-out val shards")

## 5. Phase 0-B：训练 Birkhoff `(4, 4)` `GRP_base` 并检验零和守恒门控

In [ ]:
from dataclasses import asdict
import pickle
import time
import jax

from mahjong_rl.config import GRPConfig
from mahjong_rl.models.grp import GRPNet
from mahjong_rl.trainers.run_phase0 import evaluate_grp_val
from mahjong_rl.trainers.train_grp import train_grp

grp_ckpt_path = (
    Path(CFG["PREEXISTING_CKPT_DIR"]) / "grp_phase0b.pkl"
    if CFG["PREEXISTING_CKPT_DIR"]
    else CKPT_DIR / "grp_phase0b.pkl"
)

if CFG["RUN_PHASE0"] or not grp_ckpt_path.exists():
    t0 = time.perf_counter()
    grp_cfg = GRPConfig(num_epochs=CFG["GRP_EPOCHS"], batch_size=CFG["GRP_BATCH_SIZE"])
    grp_model, grp_params, grp_hist = train_grp(train_shards, cfg=grp_cfg, seed=CFG["SEED"])
    grp_val = evaluate_grp_val(grp_model, grp_params, val_shards)
    with open(CKPT_DIR / "grp_phase0b.pkl", "wb") as f:
        pickle.dump(jax.device_get({"config": asdict(grp_cfg), "params": grp_params, "val": grp_val}), f)
    print(f"Phase 0-B finished in {time.perf_counter() - t0:.1f}s")
else:
    with open(grp_ckpt_path, "rb") as f:
        d = pickle.load(f)
    grp_cfg = GRPConfig(**d["config"])
    grp_model = GRPNet(config=grp_cfg)
    grp_params = jax.device_put(d["params"])
    grp_val = evaluate_grp_val(grp_model, grp_params, val_shards)

print("Phase 0-B Held-Out Validation Metrics:")
print(json.dumps(grp_val, indent=2))
assert grp_val["val_grp_birkhoff_mean_err"] < 1e-3, "Birkhoff mean doubly-stochastic gate failed!"
assert grp_val["val_grp_birkhoff_max_err"] < 1e-2, "Birkhoff worst-case doubly-stochastic gate failed!"
assert grp_val["val_grp_7d_zero_sum_mean_err"] < 1e-3, "7d mean zero-sum gate failed!"

## 6. Phase 0-A：训练 36× 等变 `BeliefNet` + 独立留出集温度校准 `T_cal`

In [ ]:
from mahjong_rl.config import BeliefConfig
from mahjong_rl.models.belief import BeliefNet
from mahjong_rl.trainers.run_phase0 import evaluate_belief_val
from mahjong_rl.trainers.train_belief import train_belief

bel_ckpt_path = (
    Path(CFG["PREEXISTING_CKPT_DIR"]) / "belief_phase0a.pkl"
    if CFG["PREEXISTING_CKPT_DIR"]
    else CKPT_DIR / "belief_phase0a.pkl"
)

if CFG["RUN_PHASE0"] or not bel_ckpt_path.exists():
    t0 = time.perf_counter()
    bel_cfg = BeliefConfig(num_epochs=CFG["BELIEF_EPOCHS"], batch_size=CFG["BELIEF_BATCH_SIZE"], seed=CFG["SEED"])
    bel_model, bel_params, belief_t_cal, bel_hist = train_belief(
        train_shards, cfg=bel_cfg, val_shard_paths=val_shards, suit_augment=True
    )
    bel_val = evaluate_belief_val(bel_model, bel_params, belief_t_cal, val_shards)
    with open(CKPT_DIR / "belief_phase0a.pkl", "wb") as f:
        pickle.dump(
            jax.device_get({"config": asdict(bel_cfg), "params": bel_params, "t_cal": float(belief_t_cal), "val": bel_val}),
            f,
        )
    print(f"Phase 0-A finished in {time.perf_counter() - t0:.1f}s")
else:
    with open(bel_ckpt_path, "rb") as f:
        d = pickle.load(f)
    bel_cfg = BeliefConfig(**d["config"])
    bel_model = BeliefNet(config=bel_cfg)
    bel_params = jax.device_put(d["params"])
    belief_t_cal = float(d["t_cal"])
    bel_val = evaluate_belief_val(bel_model, bel_params, belief_t_cal, val_shards)

print("Phase 0-A Held-Out Validation Metrics:")
print(json.dumps(bel_val, indent=2))

## 7. Phase 0-C：严格同构 `SovereignPolicyNet` 行为克隆（含 `beta_risk / beta_offense`）

In [ ]:
from mahjong_rl.config import BCConfig, ModelConfig
from mahjong_rl.models.equivariant import SovereignPolicyNet
from mahjong_rl.trainers.run_phase0 import evaluate_bc_val
from mahjong_rl.trainers.train_bc import train_bc

bc_ckpt_path = (
    Path(CFG["PREEXISTING_CKPT_DIR"]) / "bc_phase0c.pkl"
    if CFG["PREEXISTING_CKPT_DIR"]
    else CKPT_DIR / "bc_phase0c.pkl"
)

model_cfg = ModelConfig()
if CFG["RUN_PHASE0"] or not bc_ckpt_path.exists():
    t0 = time.perf_counter()
    bc_cfg = BCConfig(
        model=model_cfg,
        num_epochs=CFG["BC_EPOCHS"],
        batch_size=CFG["BC_BATCH_SIZE"],
        suit_augment=True,
        seed=CFG["SEED"],
    )
    pol_model, bc_params, bc_hist = train_bc(
        train_shards,
        belief_model=bel_model,
        belief_params=bel_params,
        grp_model=grp_model,
        grp_params=grp_params,
        cfg=bc_cfg,
        belief_t_cal=belief_t_cal,
    )
    bc_val = evaluate_bc_val(
        pol_model, bc_params, bel_model, bel_params, belief_t_cal, grp_model, grp_params, val_shards
    )
    with open(CKPT_DIR / "bc_phase0c.pkl", "wb") as f:
        pickle.dump(
            jax.device_get({"model_config": asdict(model_cfg), "params": bc_params, "t_cal": float(belief_t_cal), "val": bc_val}),
            f,
        )
    print(f"Phase 0-C finished in {time.perf_counter() - t0:.1f}s")
else:
    with open(bc_ckpt_path, "rb") as f:
        d = pickle.load(f)
    model_cfg = ModelConfig(**d["model_config"])
    pol_model = SovereignPolicyNet(config=model_cfg)
    bc_params = jax.device_put(d["params"])
    bc_val = evaluate_bc_val(
        pol_model, bc_params, bel_model, bel_params, belief_t_cal, grp_model, grp_params, val_shards
    )

print("Phase 0-C Held-Out Validation Metrics:")
print(json.dumps(bc_val, indent=2))

## 8. Phase 1/2 (`train_rl_v6`) + Phase 3 (`distill_sdtap`) 行为空间自蒸馏

In [ ]:
import matplotlib.pyplot as plt
from mahjong_rl.config import OracleConfig, RLConfig, RNaDConfig
from mahjong_rl.trainers.distill_sdtap import distill_sdtap
from mahjong_rl.trainers.train_rl import train_rl_v6

rl_res = None
deploy_params = bc_params
deploy_bel_params = bel_params

if CFG["RUN_PHASE12_RL"]:
    t0 = time.perf_counter()
    rl_cfg = RLConfig(
        env_name="red_4p",
        round_mode="half",
        model=model_cfg,
        belief=bel_cfg,
        grp=grp_cfg,
        oracle=OracleConfig(),
        rnad=RNaDConfig(
            warmup_updates=CFG["RL_WARMUP_UPDATES"],
            outer_reset_steps=CFG["RL_OUTER_RESET_STEPS"],
            reset_warmup_steps=20,
            ron_k_max=1024,
            audit_interval=25,
        ),
        num_envs=CFG["RL_NUM_ENVS"],
        num_steps=CFG["RL_NUM_STEPS"],
        num_updates=CFG["RL_NUM_UPDATES"],
        update_epochs=CFG["RL_UPDATE_EPOCHS"],
        minibatch_size=CFG["RL_MINIBATCH_SIZE"],
        seed=CFG["SEED"],
    )
    rl_res = train_rl_v6(
        rl_cfg,
        policy_params_init=bc_params,
        belief_params_init=bel_params,
        belief_t_cal=belief_t_cal,
        grp_params_init=grp_params,
        enable_self_healing=True,
    )
    print(f"Phase 1/2 RL finished in {(time.perf_counter() - t0)/60:.1f} min | snapshots={len(rl_res.outer_snapshots)}")

    with open(CKPT_DIR / "rl_phase2.pkl", "wb") as f:
        pickle.dump(
            jax.device_get({
                "policy_params": rl_res.policy_params,
                "belief_ema_params": rl_res.belief_ema_params,
                "oracle_params": rl_res.oracle_params,
                "t_cal": float(belief_t_cal),
                "history": rl_res.history,
            }),
            f,
        )

    deploy_params = rl_res.policy_params
    deploy_bel_params = rl_res.belief_ema_params

    # Plot RL health & convergence curves
    hist = rl_res.history
    xs = [r["update"] for r in hist]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(xs, [r["oracle_loss"] for r in hist], label="Oracle Loss")
    axes[0].plot(xs, [r["belief_loss"] for r in hist], label="Belief Loss")
    axes[0].set_title("Oracle & Belief Convergence"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(xs, [r["policy_entropy"] for r in hist], label="Policy Entropy")
    axes[1].plot(xs, [r["approx_kl"] * 10.0 for r in hist], label="10x Approx KL")
    axes[1].set_title("Policy Stability"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(xs, [r["beta_risk"] for r in hist], label="beta_risk")
    axes[2].plot(xs, [r["beta_offense"] for r in hist], label="beta_offense")
    axes[2].set_title("Disjoint Bounded Physics Gates"); axes[2].legend(); axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(CKPT_DIR / "rl_curves.png", dpi=140)
    plt.show()

if CFG["RUN_PHASE3_SDTAP"] and rl_res is not None and len(rl_res.outer_snapshots) >= 2:
    t0 = time.perf_counter()
    # Distill last up to 5 outer R-NaD snapshots in behavioral probability space
    snaps_to_distill = rl_res.outer_snapshots[-5:]
    _, deploy_params, sdtap_hist = distill_sdtap(
        val_shards + train_shards[:2],
        snaps_to_distill,
        belief_model=bel_model,
        belief_ema_params=deploy_bel_params,
        grp_model=grp_model,
        grp_base_params=grp_params,
        model_cfg=model_cfg,
        belief_t_cal=belief_t_cal,
        batch_size=CFG["SDTAP_BATCH_SIZE"],
        num_epochs=CFG["SDTAP_EPOCHS"],
        seed=CFG["SEED"],
    )
    print(f"Phase 3 SD-TAP finished in {time.perf_counter() - t0:.1f}s | hist={sdtap_hist}")
    with open(CKPT_DIR / "deploy_sdtap.pkl", "wb") as f:
        pickle.dump(
            jax.device_get({
                "model_config": asdict(model_cfg),
                "belief_config": asdict(bel_cfg),
                "grp_config": asdict(grp_cfg),
                "policy_params": deploy_params,
                "belief_ema_params": deploy_bel_params,
                "grp_base_params": grp_params,
                "t_cal": float(belief_t_cal),
                "sdtap_history": sdtap_hist,
            }),
            f,
        )

## 9. 严格固定牌山全半庄 Duplicate 1v3 评测 & 零样本多规则矩阵

In [ ]:
import mahjax
from mahjong_rl.algo.utility import W_M_LEAGUE, W_PHOENIX_7D, W_PHOENIX_10D
from mahjong_rl.config import EvalConfig
from mahjong_rl.eval.duplicate_eval import evaluate_duplicate_1v3
from mahjong_rl.eval.mjai_bridge import make_v6_policy_actor

full_report = {
    "cfg": CFG,
    "phase0b_grp_val": grp_val,
    "phase0a_belief_val": bel_val,
    "phase0c_bc_val": bc_val,
    "duplicate_1v3_eval": {},
}

if CFG["RUN_DUPLICATE_EVAL"]:
    eval_cfg = EvalConfig(
        env_name="red_4p",
        round_mode="half",
        num_walls=CFG["EVAL_WALLS"],
        max_steps=1600,
        rank_points=(90.0, 45.0, 0.0, -135.0),
        seed=2026,
    )

    candidates = [("phase0c_bc_7d", bc_params, bel_params, W_PHOENIX_7D)]
    if CFG["RUN_PHASE12_RL"]:
        candidates.extend([
            ("v6_final_7d", deploy_params, deploy_bel_params, W_PHOENIX_7D),
            ("v6_final_10d_zeroshot", deploy_params, deploy_bel_params, W_PHOENIX_10D),
            ("v6_final_mleague_zeroshot", deploy_params, deploy_bel_params, W_M_LEAGUE),
        ])
    else:
        candidates.extend([
            ("phase0c_bc_10d_zeroshot", bc_params, bel_params, W_PHOENIX_10D),
            ("phase0c_bc_mleague_zeroshot", bc_params, bel_params, W_M_LEAGUE),
        ])

    print(f"=== Duplicate 1v3 Full-Hanchan Eval ({4 * CFG['EVAL_WALLS']} games/entry vs rule_based_player) ===")
    for tag, p_pol, p_bel, w_rule in candidates:
        actor = make_v6_policy_actor(
            policy_model=pol_model,
            policy_params=p_pol,
            belief_model=bel_model,
            belief_ema_params=p_bel,
            grp_model=grp_model,
            grp_base_params=grp_params,
            w_rule=w_rule,
            opp_rules_known=False,
            belief_t_cal=belief_t_cal,
            greedy=True,
        )
        res = evaluate_duplicate_1v3(
            actor, mahjax.rule_based_player, eval_cfg, chunk_walls=CFG["EVAL_CHUNK_WALLS"]
        )
        full_report["duplicate_1v3_eval"][tag] = res
        print(
            f"  [{tag:26s}] pt={res['tenhou_pt_mean']:+.2f} ± {res['tenhou_pt_ci95']:.2f} | "
            f"rank={res['avg_rank']:.3f} | 1st={res['rank_1_rate']*100:.1f}% 4th={res['rank_4_rate']*100:.1f}% | "
            f"hora={res['hora_rate']*100:.1f}% deal_in={res['deal_in_rate']*100:.1f}% "
            f"riichi={res['riichi_rate']*100:.1f}% meld={res['meld_rate']*100:.1f}%"
        )

report_path = CKPT_DIR / "full_report.json"
report_path.write_text(json.dumps(full_report, indent=2), encoding="utf-8")
print(f"\nSaved full audit report to {report_path}")

## 10. 可选：自动备份权重与报表至 Hugging Face Hub

In [ ]:
if CFG["HF_UPLOAD_REPO"]:
    from huggingface_hub import HfApi
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from kaggle_secrets import UserSecretsClient
            hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            hf_token = None
    assert hf_token, "Set HF_TOKEN via Kaggle Secrets or environment variable to upload!"
    api = HfApi(token=hf_token)
    api.create_repo(CFG["HF_UPLOAD_REPO"], repo_type="model", exist_ok=True)
    api.upload_folder(
        folder_path=str(CKPT_DIR),
        repo_id=CFG["HF_UPLOAD_REPO"],
        repo_type="model",
    )
    print(f"Uploaded checkpoints & report to https://huggingface.co/{CFG['HF_UPLOAD_REPO']}")
else:
    print("HF_UPLOAD_REPO is empty; all checkpoints and reports remain persisted under:", CKPT_DIR)